# **Setup and Configuration**

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
sns.set_style("whitegrid")

# **Data Understanding**

**Dataset Description**

| No | Column | Description |
|---|---|---|
| 1 | Age | Usia customer |
| 2 | Gender | Jenis kelamin customer |
| 3 | Country | Negara asal customer |
| 4 | City | Kota tempat customer berada |
| 5 | Membership_Years | Lama customer terdaftar sebagai anggota (dalam tahun) |
| 6 | Login_Frequency | Frekuensi login customer ke platform |
| 7 | Session_Duration_Avg | Rata-rata durasi sesi penggunaan platform |
| 8 | Pages_Per_Session | Rata-rata jumlah halaman yang dibuka dalam satu sesi |
| 9 | Cart_Abandonment_Rate | Persentase keranjang belanja yang ditinggalkan tanpa checkout |
| 10 | Wishlist_Items | Jumlah produk yang disimpan di wishlist |
| 11 | Total_Purchases | Total pembelian yang telah dilakukan |
| 12 | Average_Order_Value | Rata-rata nilai transaksi per pesanan |
| 13 | Days_Since_Last_Purchase | Jumlah hari sejak pembelian terakhir |
| 14 | Discount_Usage_Rate | Tingkat penggunaan diskon saat bertransaksi |
| 15 | Returns_Rate | Tingkat pengembalian produk |
| 16 | Email_Open_Rate | Persentase email yang dibuka |
| 17 | Customer_Service_Calls | Jumlah kontak ke customer service |
| 18 | Product_Reviews_Written | Jumlah review produk yang ditulis |
| 19 | Social_Media_Engagement_Score | Skor keterlibatan pada media sosial |
| 20 | Mobile_App_Usage | Tingkat penggunaan aplikasi mobile |
| 21 | Payment_Method_Diversity | Jumlah variasi metode pembayaran yang digunakan |
| 22 | Lifetime_Value | Estimasi total nilai customer selama menggunakan layanan |
| 23 | Credit_Balance | Jumlah saldo kredit yang dimiliki |
| 24 | Churned | Status churn, dengan nilai 0 untuk tidak churn dan 1 untuk churn |
| 25 | Signup_Quarter | Quarter saat customer pertama kali mendaftar (Q1, Q2, Q3, Q4) |

**Data Loading**

In [ ]:
drive.mount('/content/drive')
path = "/content/drive/MyDrive/ecommerce_customer_churn_dataset.csv"

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Data type sebelum dibaca semua sebagai string
df_raw = pd.read_csv(path)

print("===== RAW DATA TYPES =====")
print(df_raw.dtypes)

# Membaca semua data sebagai string
df = pd.read_csv(path, dtype=str)
initial_rows, initial_cols = df.shape

**Initial Data Profiling**

Dataset Preview

In [ ]:
print("Shape awal:", df.shape)
display(df.head())

Dataset Structure

In [ ]:
df.info()

Descriptive Statistics

In [ ]:
df.describe()

Missing Value Analysis

In [ ]:
print(df.isnull().sum())

Duplicate Analysis

In [ ]:
print("Duplicate count:", df.duplicated().sum())

# **Data Cleaning**

**Helper**

In [ ]:
row_log = []
invalid_log = []

def log_row_change(step_name, before_rows, after_rows):
    row_log.append({
        "Step": step_name,
        "Rows Before": before_rows,
        "Rows After": after_rows,
        "Rows Removed": before_rows - after_rows
    })

def log_invalid_change(step_name, col_name, invalid_count):
    invalid_log.append({
        "Step": step_name,
        "Column": col_name,
        "Invalid Count": int(invalid_count)
    })

Handling Inconsistencies

In [ ]:
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace(['nan', 'None', ''], np.nan)

Correcting Data Types

In [ ]:
numeric_cols_expected = [
    'Age', 'Membership_Years', 'Login_Frequency', 'Session_Duration_Avg',
    'Pages_Per_Session', 'Cart_Abandonment_Rate', 'Wishlist_Items',
    'Total_Purchases', 'Average_Order_Value', 'Days_Since_Last_Purchase',
    'Discount_Usage_Rate', 'Returns_Rate', 'Email_Open_Rate',
    'Customer_Service_Calls', 'Product_Reviews_Written',
    'Social_Media_Engagement_Score', 'Mobile_App_Usage',
    'Payment_Method_Diversity', 'Lifetime_Value', 'Credit_Balance',
    'Churned'
]

In [ ]:
def clean_numeric_super_strict(x, col_name=None):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    if s == "":
        return np.nan

    # Hapus persen, spasi, koma
    s = s.replace('%', '').replace(' ', '').replace(',', '')

    # Hanya ada digit, titik, minus
    s = re.sub(r'[^0-9.\-]', '', s)

    if s in {"", "-", ".", "-."}:
        return np.nan

    # Kalo titik > 1, ambil angka sebelum titik ke2 aja
    if s.count('.') > 1:
        parts = s.split('.')

        if len(parts) >= 2 and parts[0].isdigit() and parts[1].isdigit():
            s = parts[0] + '.' + parts[1]
        else:
            return np.nan

    try:
        val = float(s)
    except:
        return np.nan

    # Filter masing-masing kolom
    if col_name is not None:
        if col_name == 'Session_Duration_Avg':
            if val < 0 or val > 1000:
                return np.nan

        elif col_name == 'Average_Order_Value':
            if val < 0 or val > 10000:
                return np.nan

        elif col_name == 'Discount_Usage_Rate':
            if val < 0 or val > 100:
                return np.nan

        elif col_name == 'Returns_Rate':
            if val < 0 or val > 100:
                return np.nan

        elif col_name == 'Mobile_App_Usage':
            if val < 0 or val > 24:
                return np.nan

        elif col_name == 'Lifetime_Value':
            if val < 0 or val > 1_000_000:
                return np.nan

    return val

In [ ]:
for col in numeric_cols_expected:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: clean_numeric_super_strict(x, col))

In [ ]:
categorical_cols = [col for col in df.columns if col not in numeric_cols_expected]

for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace(['nan', 'None', ''], np.nan)

Drop Irrelevant for EDA

In [ ]:
missing_ratio = df.isnull().mean()
cols_drop_missing = missing_ratio[missing_ratio > 0.5].index.tolist()

if len(cols_drop_missing) > 0:
    print("\nKolom dihapus karena missing > 50%:", cols_drop_missing)

df = df.drop(columns=cols_drop_missing)

Duplicate Handling

In [ ]:
before_rows = df.shape[0]
df = df.drop_duplicates()
after_rows = df.shape[0]
log_row_change("Drop duplicates", before_rows, after_rows)

Important Missing Handling

In [ ]:
if 'Age' in df.columns:
    before_rows = df.shape[0]
    df = df.dropna(subset=['Age'])
    after_rows = df.shape[0]
    log_row_change("Drop rows with missing Age", before_rows, after_rows)

Inconsistent Value Handling

In [ ]:
logical_limits = {
    'Age': (10, 100),
    'Membership_Years': (0, 50),
    'Login_Frequency': (0, 1000),
    'Session_Duration_Avg': (0, 1000),
    'Pages_Per_Session': (0, 50),
    'Cart_Abandonment_Rate': (0, 100),
    'Wishlist_Items': (0, 200),
    'Total_Purchases': (0, 500),
    'Average_Order_Value': (0, 10000),
    'Days_Since_Last_Purchase': (0, 3650),
    'Discount_Usage_Rate': (0, 100),
    'Returns_Rate': (0, 100),
    'Email_Open_Rate': (0, 100),
    'Customer_Service_Calls': (0, 100),
    'Product_Reviews_Written': (0, 1000),
    'Social_Media_Engagement_Score': (0, 100),
    'Mobile_App_Usage': (0, 24),
    'Payment_Method_Diversity': (0, 20),
    'Lifetime_Value': (0, 1_000_000),
    'Credit_Balance': (0, 100000),
    'Churned': (0, 1)
}

print("\n===== INVALID VALUE CHECK =====")
for col, (low, high) in logical_limits.items():
    if col in df.columns:
        invalid_mask = (df[col] < low) | (df[col] > high)
        invalid_count = invalid_mask.sum()
        print(f"{col}: {invalid_count} values")
        log_invalid_change("Logical limits -> set NaN", col, invalid_count)
        df.loc[invalid_mask, col] = np.nan

Strict Filtering

In [ ]:
special_caps = {
    'Session_Duration_Avg': 500,
    'Average_Order_Value': 5000,
    'Discount_Usage_Rate': 100,
    'Returns_Rate': 100,
    'Mobile_App_Usage': 24,
    'Lifetime_Value': 500000
}

print("\n===== EXTRA STRICT FILTER =====")
for col, cap in special_caps.items():
    if col in df.columns:
        bad_mask = df[col] > cap
        bad_count = bad_mask.sum()
        print(f"{col}: {bad_count} values")
        log_invalid_change("Special caps -> set NaN", col, bad_count)
        df.loc[bad_mask, col] = np.nan

Checking Problematic Columns

In [ ]:
problem_cols = [
    'Session_Duration_Avg',
    'Average_Order_Value',
    'Discount_Usage_Rate',
    'Returns_Rate',
    'Mobile_App_Usage',
    'Lifetime_Value',
    'Cart_Abandonment_Rate',
    'Total_Purchases'
]

display(df[problem_cols].describe(percentiles=[0.95, 0.99]).T)

Missing Value Imputation

In [ ]:
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            mode_series = df[col].mode(dropna=True)
            if len(mode_series) > 0:
                df[col] = df[col].fillna(mode_series.iloc[0])
            else:
                df[col] = df[col].fillna("Unknown")

Robust Outlier Clipping

In [ ]:
skip_clip = {
    'Churned',
    'Cart_Abandonment_Rate',
    'Discount_Usage_Rate',
    'Returns_Rate',
    'Email_Open_Rate',
    'Mobile_App_Usage'
}

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

for col in numeric_cols:
    if col in skip_clip:
        continue

    series = df[col].dropna()

    if len(series) < 10:
        continue

    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1

    if IQR == 0:
        continue

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    if col in logical_limits:
        logical_low, logical_high = logical_limits[col]
        lower = max(lower, logical_low)
        upper = min(upper, logical_high)

    df[col] = np.clip(df[col], lower, upper)

Final Formatting

In [ ]:
integer_like_cols = [
    'Age', 'Wishlist_Items',
    'Total_Purchases', 'Days_Since_Last_Purchase', 'Customer_Service_Calls',
    'Product_Reviews_Written', 'Payment_Method_Diversity', 'Churned'
]

for col in integer_like_cols:
    if col in df.columns:
        df[col] = df[col].round(0).astype(int)

decimal_rules = {
    'Membership_Years': 1,
    'Login_Frequency': 1,
    'Session_Duration_Avg': 1,
    'Average_Order_Value': 3,
    'Discount_Usage_Rate': 2,
    'Returns_Rate': 2,
    'Mobile_App_Usage': 1,
    'Lifetime_Value': 3,
    'Cart_Abandonment_Rate': 2,
    'Email_Open_Rate': 2,
    'Pages_Per_Session': 2,
    'Credit_Balance': 2,
    'Social_Media_Engagement_Score': 2
}

for col, n_decimals in decimal_rules.items():
    if col in df.columns:
        df[col] = df[col].round(n_decimals)

if 'Churned' in df.columns:
    df['Churned'] = df['Churned'].clip(0, 1).round(0).astype(int)

Feature Adding

In [ ]:
if 'Total_Purchases' in df.columns and 'Average_Order_Value' in df.columns:
    df['Estimated_Revenue'] = (df['Total_Purchases'] * df['Average_Order_Value']).round(3)

# **Data Profiling After Cleaning**

Final Checking

In [ ]:
df.info()

In [ ]:
final_rows, final_cols = df.shape

print("\n===== FINAL CHECK UMUM =====")
print("Shape awal:", (initial_rows, initial_cols))
print("Shape akhir:", df.shape)
print("Baris terhapus total:", initial_rows - final_rows)
print("Kolom terhapus total:", initial_cols - final_cols)
print("Missing akhir:", df.isnull().sum().sum())

display(df.describe(include='all'))

Summary

In [ ]:
row_log_df = pd.DataFrame(row_log)

print("\n===== RINGKASAN PENGHAPUSAN BARIS =====")
if len(row_log_df) > 0:
    display(row_log_df)
    print("Total baris terhapus:", row_log_df["Rows Removed"].sum())
else:
    print("Tidak ada baris yang dihapus.")

In [ ]:
invalid_log_df = pd.DataFrame(invalid_log)

print("\n===== RINGKASAN INVALID VALUE =====")
if len(invalid_log_df) > 0:
    display(invalid_log_df)
else:
    print("Tidak ada invalid value yang terdeteksi.")

# **Save Output**

In [ ]:
local_path = "/content/final_preprocessed_data.csv"
df.to_csv(local_path, index=False)

print("\nPreprocessing selesai!")
print("File disimpan di:", local_path)

from google.colab import files
files.download(local_path)

# **Descriptive Analysis**

**Prepare EDA Dataset**

In [ ]:
df_eda = df.copy()
numerical_cols = df_eda.select_dtypes(include=['number']).columns # Hanya untuk statistic
cat_cols = ['Gender', 'Country', 'Signup_Quarter']
num_cols = [
    'Age',
    'Login_Frequency',
    'Session_Duration_Avg',
    'Pages_Per_Session',
    'Cart_Abandonment_Rate',
    'Total_Purchases',
    'Average_Order_Value',
    'Estimated_Revenue',
    'Days_Since_Last_Purchase',
    'Email_Open_Rate',
    'Customer_Service_Calls',
    'Lifetime_Value'
]

**Descriptive Statistics**

In [ ]:
descriptive_stats = df_eda[numerical_cols].describe()

variance_values = df_eda[numerical_cols].var()
descriptive_stats.loc['variance'] = variance_values

display(descriptive_stats)

**Univariate**

Distribution of Categorical Features

In [ ]:
def check_univariate_categorical(df, cols):
    cols = [col for col in cols if col in df.columns]

    plt.figure(figsize=(18, 4))

    for i, col in enumerate(cols, 1):
        plt.subplot(1, len(cols), i)

        counts = df[col].value_counts()

        colors = ['tab:orange' if x == counts.max() else 'tab:blue' for x in counts.values]

        bars = plt.bar(counts.index, counts.values, color=colors)

        plt.title(col)

        plt.grid(False)

        plt.ylim(0, counts.max() * 1.1)

        for bar in bars:
            height = bar.get_height()
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                height + counts.max() * 0.01,
                f'{int(height):,}',
                ha='center',
                va='bottom',
                fontsize=9
            )

        if col == 'Country':
            plt.xticks(rotation=45)
        else:
          plt.xticks(rotation=0)

    plt.tight_layout()
    plt.show()

check_univariate_categorical(df_eda, cat_cols)

Distribution of Numerical Features

In [ ]:
def check_univariate_numerical(df, cols):
    cols = [col for col in cols if col in df.columns]

    n = len(cols)
    rows = -(-n // 3)

    plt.figure(figsize=(18, 4 * rows))

    for i, col in enumerate(cols, 1):
        plt.subplot(rows, 3, i)

        sns.histplot(df[col], kde=True, bins=30)

        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Frequency')

    plt.tight_layout()
    plt.show()

check_univariate_numerical(df_eda, num_cols)

**Bivariate**

Distribution of Categorical Features

In [ ]:
def check_bivariate_categorical(df, cols, target):
    cols = [col for col in cols if col in df.columns]

    if target not in df.columns:
        print(f"Kolom target '{target}' tidak ada di dataframe.")
        return

    n = len(cols)
    rows = -(-n // 3)

    plt.figure(figsize=(18, 4 * rows))

    for i, col in enumerate(cols, 1):
        plt.subplot(rows, 3, i)

        ctab = pd.crosstab(df[col], df[target])
        ctab = ctab.loc[ctab.sum(axis=1).sort_values(ascending=False).index]

        ctab.plot(
            kind='bar',
            stacked=True,
            ax=plt.gca(),
            color=['#1f77b4', '#ff7f0e'],
            width=0.6
        )

        plt.grid(False)

        plt.title(f'{col} vs {target}')
        plt.xlabel(col)
        plt.ylabel('Count')

        if col in ['Gender', 'Signup_Quarter']:
            plt.xticks(rotation=0)
        elif df[col].nunique() > 5:
            plt.xticks(rotation=45)

        plt.legend(title=target)

    plt.tight_layout()
    plt.show()

check_bivariate_categorical(df_eda, cat_cols, 'Churned')

Distribution of Numerical Features

In [ ]:
def check_bivariate_numerical(df, cols, target):
    cols = [col for col in cols if col in df.columns]

    if target not in df.columns:
        print(f"Kolom target '{target}' tidak ada di dataframe.")
        return

    n = len(cols)
    rows = -(-n // 3)

    plt.figure(figsize=(18, 4 * rows))

    for i, col in enumerate(cols, 1):
        plt.subplot(rows, 3, i)

        sns.boxplot(x=df[target], y=df[col])

        plt.title(f'{col} vs {target}')
        plt.xlabel(target)
        plt.ylabel(col)

    plt.tight_layout()
    plt.show()

check_bivariate_numerical(df_eda, num_cols, 'Churned')

**Correlation Analysis**

In [ ]:
def plot_correlation_heatmap(df, cols):
    corr_matrix = df[cols].corr()

    plt.figure(figsize=(16, 10))
    sns.heatmap(
        corr_matrix,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        vmin=-1,
        vmax=1
    )
    plt.title("Correlation Heatmap", fontsize=16)
    plt.tight_layout()
    plt.show()

plot_correlation_heatmap(df_eda, numerical_cols)